# Dense Patch Token Evaluation — Notebook

Demonstrates both `dpt` evaluation wirings side by side:

- **Part 1 — Tile evaluation** (`run_dpt_eval`) — each entry is a crop of a larger image.
- **Part 2 — Whole-image evaluation** (`run_dpt_image_eval`) — each entry is one full (untiled) image.

Both wirings apply the exact same rules and compute the exact same metrics — the only difference is field naming (`tiles`/`per_tile` vs. `images`/`per_image`). Part 1 uses real (non-synthetic) feature maps shipped under `tests/data/dpt_tiles/`, pooled down to a small grid to keep the notebook lightweight, together with the real ground-truth masks already shipped in `tests/data/masks/`.

Run all cells top-to-bottom from the `examples/dpt/` directory.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

from precisionai.agrieval.dpt import load_tile_placement, load_tiles, print_result, run_dpt_eval, run_dpt_image_eval

_REPO_ROOT = Path("../..").resolve()
_DPT_TILES_DIR = _REPO_ROOT / "tests" / "data" / "dpt_tiles"
_MASKS_DIR = _REPO_ROOT / "tests" / "data" / "masks"
_CLASSES_PATH = _REPO_ROOT / "tests" / "data" / "class_map.json"


def _rel(path: Path) -> Path:
    """Display a repo path relative to the repo root, never as an absolute local path."""
    return path.relative_to(_REPO_ROOT)

---
## Part 1 — Tile evaluation

Each entry represents a **crop** of a larger source image. This notebook loads real (non-synthetic) feature maps shipped under `tests/data/dpt_tiles/` — one archive per source image, each holding that image's own real tiles — pooled down to a small patch grid to keep the fixtures lightweight, with the real embedding dimension and tile pixel geometry preserved. Ground truth comes from the real masks already shipped in `tests/data/masks/`, enabling the label-aware metrics (`classes`, `per_class`, `knn_confusion`, `separation`) out of the box.

The `separation` block holds the split-free label separation metrics — silhouette (cosine), Calinski-Harabasz, k-means ARI/NMI, and PC1-AUROC (foreground vs. the `background` class) — computed without any train/eval split. A metric whose preconditions fail (e.g. a single class present) is reported as `None` with an explanatory warning.

In [2]:
tiles = {}
tile_placement = {}
for archive in sorted(_DPT_TILES_DIR.rglob("*.npz")):
    tiles.update(load_tiles(archive))
    tile_placement.update(load_tile_placement(archive))

n_images = len({p.image_stem for p in tile_placement.values()})
print(f"Tiles: {len(tiles)} real tile(s) across {n_images} source image(s)")
print(f"Masks: {_rel(_MASKS_DIR)}")
print(f"Classes: {_rel(_CLASSES_PATH)}")

Tiles: 16 real tile(s) across 4 source image(s)
Masks: tests\data\masks
Classes: tests\data\class_map.json


In [3]:
tile_result = run_dpt_eval(tiles=tiles, tile_placement=tile_placement, masks_dir=_MASKS_DIR, classes_path=_CLASSES_PATH)

print_result(tile_result)

k-NN graph:   0%|          | 0/1536 [00:00<?, ?item/s]

n_entries: 16
embed_dim: 384
grid     : 8x12
n_patches: 1536
k_values : [5, 10, 20]
classes  : ['background', 'Crop | Soybean', 'Weed | Weed']

── global_metrics ──────────────────────────────────────────────────────
  effective_rank        : 17.4880  (ratio=0.0455  dim=384)
  pca_explained_variance: pc1=0.1276  top_10=0.6720  top_50=0.9217  top_100=0.9656
  pairwise cosine       : mean=0.3170  std=0.1725  (p05=0.0649  p50=0.3015  p95=0.6246)
  centroid cosine       : mean=0.5637  std=0.1014  norm=0.5637
  uniformity            : -2.4655
  mean_patch_smoothness : 0.7209
  mean_outlier_fraction : 0.0007

── per_class ───────────────────────────────────────────────────────────
  background           n_patches=1099  effective_rank=17.3190  pc1=0.1403
  Crop | Soybean       n_patches=430  effective_rank=8.5312  pc1=0.2472
  Weed | Weed          n_patches=7  effective_rank=2.1188  pc1=0.6475

── knn_confusion ───────────────────────────────────────────────────────
  purity@5  : mean=0.9159 

---
## Part 2 — Whole-image evaluation

Each entry represents one **whole, untiled image** rather than a crop. The wiring requires the exact same `(P, H, W)` shape contract as tiles, since a single evaluation run is always against one backbone/tiling configuration. We reuse the same real feature maps here as a stand-in for whole images, passed through `run_dpt_image_eval` and keyed as `images` instead of `tiles` — unsupervised metrics only in this part, since these entries are tile crops rather than genuine whole-image ground truth (already demonstrated with real masks in Part 1).

In [4]:
images = tiles  # same real feature maps, viewed as whole images instead of tile crops

image_result = run_dpt_image_eval(images=images)

print_result(image_result)

n_entries: 16
embed_dim: 384
grid     : 8x12
n_patches: 1536
k_values : [5, 10, 20]
classes  : None

── global_metrics ──────────────────────────────────────────────────────
  effective_rank        : 17.4880  (ratio=0.0455  dim=384)
  pca_explained_variance: pc1=0.1276  top_10=0.6720  top_50=0.9217  top_100=0.9656
  pairwise cosine       : mean=0.3170  std=0.1725  (p05=0.0649  p50=0.3015  p95=0.6246)
  centroid cosine       : mean=0.5637  std=0.1014  norm=0.5637
  uniformity            : -2.4655
  mean_patch_smoothness : 0.7209
  mean_outlier_fraction : 0.0007


### Parity check

Given the same input feature maps, the two wirings' unsupervised computation must agree exactly — modulo their field naming. (Label-aware fields aren't compared here since Part 2 intentionally ran without ground truth — see above.)

In [5]:
import math
from typing import Any


def _approx_equal(a: Any, b: Any, *, rel_tol: float = 1e-6, abs_tol: float = 1e-9) -> bool:
    """Recursively compare nested dict/list/number results.

    Tolerant of floating-point noise (e.g. from PCA solvers) rather than
    requiring bit-identical recomputation.
    """
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_approx_equal(a[k], b[k], rel_tol=rel_tol, abs_tol=abs_tol) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(
            _approx_equal(x, y, rel_tol=rel_tol, abs_tol=abs_tol) for x, y in zip(a, b, strict=True)
        )
    if isinstance(a, int | float) and isinstance(b, int | float):
        return math.isclose(a, b, rel_tol=rel_tol, abs_tol=abs_tol)
    return a == b


def _strip_noisy_pca_tail(metrics: dict[str, Any]) -> dict[str, Any]:
    """Drop the raw per-component PCA list.

    Near-zero components are numerically unstable across separate PCA
    solves and aren't a meaningful equality check.
    """
    sanitized = dict(metrics)
    if "pca_explained_variance" in sanitized:
        pca = dict(sanitized["pca_explained_variance"])
        pca.pop("explained_variance_ratio", None)
        sanitized["pca_explained_variance"] = pca
    return sanitized


checks = {
    "n_images == n_tiles": image_result["n_images"] == tile_result["n_tiles"],
    "image_ids == tile_ids": image_result["image_ids"] == tile_result["tile_ids"],
    "global_metrics match": _approx_equal(
        _strip_noisy_pca_tail(image_result["global_metrics"]),
        _strip_noisy_pca_tail(tile_result["global_metrics"]),
        rel_tol=1e-4,
    ),
    "per_image == per_tile": _approx_equal(image_result["per_image"], tile_result["per_tile"]),
}
failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise ValueError(f"Tile/image wiring parity check failed: {failed}")

print("Tile and image wirings agree exactly (modulo field naming and floating-point noise).")

Tile and image wirings agree exactly (modulo field naming and floating-point noise).
